# Geographic plots with `GeoPlot`

<style>
blockquote:has(.notebook-admonition-title) {
  --notebook-admonition-color: var(--color-admonition-title--note, #087fc7);
  --notebook-admonition-title-background:
    var(--color-admonition-title-background--note, rgba(8, 127, 199, 0.18));
  background: var(--color-admonition-background, transparent);
  border: 0;
  border-left: 0.2rem solid var(--notebook-admonition-color);
  border-radius: 0.2rem;
  box-shadow: 0 0.2rem 0.5rem rgba(0, 0, 0, 0.05), 0 0 0.0625rem rgba(0, 0, 0, 0.1);
  font-size: var(--admonition-font-size, 0.8125rem);
  margin: 1rem auto;
  overflow: hidden;
  padding: 0 0.5rem 0.5rem;
}
blockquote p:has(> .notebook-admonition-title) {
  background: var(--notebook-admonition-title-background);
  font-size: var(--admonition-title-font-size, 0.8125rem);
  font-weight: 500;
  line-height: 1.3;
  margin: 0 -0.5rem 0.5rem;
  padding: 0.4rem 0.5rem 0.4rem 2rem;
  position: relative;
}
blockquote p:has(> .notebook-admonition-title)::before {
  color: var(--notebook-admonition-color);
  content: "✎";
  left: 0.65rem;
  position: absolute;
}
.notebook-admonition-title {
  font-weight: inherit;
}
table:not(.dataframe) {
  border: 1px solid var(--docs-hairline, rgba(128, 128, 128, 0.35));
  border-collapse: collapse;
}
table:not(.dataframe) th,
table:not(.dataframe) td {
  border: 1px solid var(--docs-hairline, rgba(128, 128, 128, 0.35));
}
</style>

<div style="text-align: center;"><a class="sd-sphinx-override sd-btn sd-text-wrap sd-btn-primary reference external" href="https://github.com/mggg/gerrytools/tree/main/user_guide/_static/data">Browse tutorial data</a></div>

`GeoPlot` draws district plans, choropleths, outlines, highlights, markers, and labels as
layers on a GeoDataFrame. This page shows each layer separately before combining a few of
them at the end.


In [ ]:
from pathlib import Path

import geopandas as gpd
import pandas as pd

from gerrytools.plotting import GeoPlot

precincts = gpd.read_file(Path("data/ga_2016_precincts.gpkg"))
precincts[["BVAP", "VAP"]] = precincts[["BVAP", "VAP"]].apply(pd.to_numeric)
precincts["BVAP_SHARE"] = precincts["BVAP"].div(precincts["VAP"])

## Districting plans

Pass the assignment column to `add_districting_plan_layer()`. The values are treated as
categories, and the default Districtr palette assigns one color to each district label.


In [ ]:
plan = GeoPlot(precincts)
plan.add_districting_plan_layer("CD")
plan.show()

The first map still draws each precinct boundary. Set `dissolve=True` to union precincts with
the same assignment before drawing them, and `show_labels=True` to place one label on each
resulting district. Dissolving changes this layer only; the source GeoDataFrame remains at
precinct resolution.


In [ ]:
labeled_plan = GeoPlot(precincts)
labeled_plan.add_districting_plan_layer("CD", dissolve=True, show_labels=True)
labeled_plan.show()

## Choropleths

Pass a numeric column to `add_choropleth_layer()`. Unlike a districting-plan layer, a
choropleth maps values through a continuous colormap. By default the scale spans the observed
minimum and maximum; use `vmin` and `vmax` when several maps need a common scale.


In [ ]:
choropleth = GeoPlot(precincts)
choropleth.add_choropleth_layer("BVAP_SHARE")
choropleth.show()

A colorbar is one additional argument. Its default label is the source column name. The
shared-controls guide shows how to change its placement and formatting.


In [ ]:
with_colorbar = GeoPlot(precincts)
with_colorbar.add_choropleth_layer("BVAP_SHARE", show_colorbar=True)
with_colorbar.show()

## Outlines

An outline contributes boundaries without a fill. `dissolve_column=` groups the source rows
before drawing, so the precinct geometries below become county outlines. Omit it when the
source rows are already the units that should be outlined.


In [ ]:
counties = GeoPlot(precincts)
counties.add_outline_layer(dissolve_column="CTYNAME")
counties.show()

## Highlights

A highlight unions the selected geometries into one translucent shape. A boolean Series is a
natural mask when the selected area is already described by a GeoDataFrame column.


In [ ]:
highlighted = GeoPlot(precincts)
highlighted.add_highlight_layer(geometry_mask=precincts["CTYNAME"].eq("Fulton"))
highlighted.show()

## Markers

Markers can come from a point GeoSeries or from `(latitude, longitude)` pairs. Latitude and
longitude default to EPSG:4326 and are reprojected to the map's coordinate reference system
when the layer renders.


In [ ]:
markers = GeoPlot(precincts)
markers.add_marker_layer(latlon_list=[(33.749, -84.388)])
markers.show()

## Focus the map

`focus_axes()` fits the axes to a geometry or boolean mask without filtering the plotted data.
Here the statewide plan remains intact while the visible window is fit to metro Atlanta.


In [ ]:
fulton_mask = precincts["CTYNAME"].eq("Fulton")
metro = GeoPlot(precincts)
metro.add_districting_plan_layer("CD")
metro.add_outline_layer(geometry_mask=fulton_mask, dissolve_column="CTYNAME", edgewidth=2)
metro.focus_axes(geometry_mask=fulton_mask)
metro.show()

## Combine layers

A composed map is still a sequence of ordinary layer calls. This example combines a
choropleth, district outlines, and a focused extent. `default_outline=False` removes the base
precinct outline because the explicit district outline supplies the boundaries needed here.


In [ ]:
combined = GeoPlot(precincts, default_outline=False)
combined.add_choropleth_layer("BVAP_SHARE", vmin=0, vmax=1)
combined.add_outline_layer(dissolve_column="CD")
combined.focus_axes(geometry_mask=fulton_mask)
combined.show()

## More options

- [Geographic plot controls](options.ipynb) covers label styles, manual
  label placement, colorbars, layer styling, and output.
- [Geographic workflow](workflow.ipynb) builds a multi-layer report figure.
- [Plotting API](../../../api/plotting.rst) lists every layer method and option object.
